# Getting Started: A First Steady Simulation

This tutorial walks through a first steady aerodynamic simulation with Ptera
Software. You will build a simple airplane from scratch, one object at a time:

1. Airfoil
2. WingCrossSection
3. Wing
4. Airplane
5. OperatingPoint
6. SteadyProblem
7. Solver

By the end you will have solved for the lift and induced drag of a rectangular
wing and seen how to log, save, and plot the results.

This tutorial uses the steady horseshoe vortex lattice method (VLM). Ptera
Software also ships steady ring VLM, unsteady ring VLM (UVLM), aeroelastic, and
free flight solvers; the object model you learn here is shared by all of them.


## Install

Install the package in your environment if you have not already:

```shell
pip install pterasoftware ipywidgets
```

Then import the package. The examples in this tutorial assume you are working
in a Jupyter notebook or an interactive Python session.


In [1]:
import pterasoftware as ps

## 1. Airfoil

An `Airfoil` defines the two-dimensional shape of a wing cross section. Ptera
Software ships a large database of airfoils (courtesy of the UIUC Airfoil
Coordinates Database); you select one by name.

Here we use the classic NACA 2412 airfoil:


In [2]:
airfoil = ps.geometry.airfoil.Airfoil(name="naca2412")

## 2. WingCrossSection

A `WingCrossSection` places an `Airfoil` at a location along the wing and
controls how that station is discretized into panels. `num_spanwise_panels`
controls the panel resolution in the spanwise direction, and `chord` sets the
local chord length.

A wing is built from two or more cross sections. Here we define a root station
and a tip station. The tip is offset from the root with `Lp_Wcsp_Lpp`: the
second component is the spanwise offset, the first is the streamwise offset
(sweep), and the third is the vertical offset (dihedral).


In [3]:
root_wing_cross_section = ps.geometry.wing_cross_section.WingCrossSection(
    airfoil=airfoil,
    num_spanwise_panels=8,
    chord=1.75,
    Lp_Wcsp_Lpp=(0.0, 0.0, 0.0),
    control_surface_symmetry_type="symmetric",
)

In [4]:
tip_wing_cross_section = ps.geometry.wing_cross_section.WingCrossSection(
    airfoil=airfoil,
    num_spanwise_panels=None,
    chord=1.5,
    Lp_Wcsp_Lpp=(0.75, 6.0, 1.0),
    control_surface_symmetry_type="symmetric",
)

## 3. Wing

A `Wing` groups the cross sections into a lifting surface and discretizes it
into panels in the chordwise direction (`num_chordwise_panels`).

Setting `symmetric=True` mirrors the wing about the plane defined by
`symmetryNormal_G` and `symmetryPoint_G_Cg`. The wing cross sections are the
starboard half, and the mirror adds the port half.


In [5]:
wing = ps.geometry.wing.Wing(
    wing_cross_sections=[root_wing_cross_section, tip_wing_cross_section],
    symmetric=True,
    symmetryNormal_G=(0.0, 1.0, 0.0),
    symmetryPoint_G_Cg=(0.0, 0.0, 0.0),
    num_chordwise_panels=6,
)

## 4. Airplane

An `Airplane` is a collection of one or more `Wing` objects. This is the top
level of the geometry hierarchy.


In [6]:
airplane = ps.geometry.airplane.Airplane(
    wings=[wing],
)

## 5. OperatingPoint

An `OperatingPoint` describes the flight condition: free stream velocity,
angle of attack, sideslip, air density, and more. The default values below are
a typical low-speed cruise condition.


In [7]:
operating_point = ps.operating_point.OperatingPoint()

## 6. SteadyProblem

A `SteadyProblem` bundles the geometry and the operating point into a single
object that the solver can act on.


In [8]:
problem = ps.problems.SteadyProblem(
    airplanes=[airplane],
    operating_point=operating_point,
)

## 7. SteadyHorseshoeVortexLatticeMethodSolver

The `SteadyHorseshoeVortexLatticeMethodSolver` solves the problem. Its `run`
method builds the vortex lattice and solves the linear system, which takes less
than a second for this geometry thanks to JIT compilation and parallelization.


In [9]:
solver = (
    ps.steady_horseshoe_vortex_lattice_method.SteadyHorseshoeVortexLatticeMethodSolver(
        steady_problem=problem,
    )
)
solver.run()

## 8. Results

The solver stores the results directly on the `Airplane` object.
Common quantities are the total lift and induced drag coefficients:


In [10]:
print("Lift coefficient:", airplane.liftCoefficient_W)
print("Induced drag coefficient:", airplane.inducedDragCoefficient_W)
print("Force vector (wind axes, N):", airplane.forces_W)

Lift coefficient: 0.5548759749558987
Induced drag coefficient: 0.012512581250986098
Force vector (wind axes, N): [-1.49447142e+01 -6.31439345e-16 -6.62729993e+02]


Per-panel data is available on `solver.panels`, which is useful for
post-processing such as spanwise load plots.


In [11]:
print("Number of panels:", len(solver.panels))
print("Panel object:", solver.panels[0])

Number of panels: 96
Panel object: <pterasoftware._panel.Panel object at 0x778a02cea200>


## 9. Visualization

The `output` module's `draw` function renders the solved airplane with PyVista. Here it colors the panels by their lift coefficients and shows the streamlines trailing the wing. A window opens so you can orient the view, and pressing any key closes it. Setting `save=True` then writes the view to a WebP file next to this notebook, which the markdown cell after the code embeds, so the render appears both when you run the notebook and on the documentation website:


In [12]:
ps.output.draw(
    solver=solver,
    scalar_type="lift",
    show_streamlines=True,
    save=True,
    path="getting_started_draw.webp",
)

![The tutorial wing colored by lift coefficient, with streamlines trailing the wing.](getting_started_draw.webp)


## 10. Logging

The `output` module can write a formatted log of the simulation setup and
results. Call `ps.set_up_logging` once to configure logging, then use
`ps.output.log_results`:


In [13]:
ps.set_up_logging(level="Info")
ps.output.log_results(solver)

INFO    |output                                      |Airplane "Untitled Airplane":
INFO    |output                                      |  Reynolds Number: 1.08E+06
INFO    |output                                      |  Forces (in Geometry Axes):
INFO    |output                                      |    fX_G:           -42.9 N
INFO    |output                                      |    fY_G:       -6.31E-16 N
INFO    |output                                      |    fZ_G:            662. N
INFO    |output                                      |  Moments (in Geometry Axes, Relative to the CG):
INFO    |output                                      |    mX_G_Cg:     7.31E-14 Nm
INFO    |output                                      |    mY_G_Cg:        -617. Nm
INFO    |output                                      |    mZ_G_Cg:    -1.17E-15 Nm
INFO    |output                                      |  Force Coefficients (in Geometry Axes):
INFO    |output                                      |   

## 11. Saving and loading

Solved simulations can be saved to a .psz file (a zip archive of JSON members) and reloaded without re-running,
which is handy for sharing or post-processing later.


In [14]:
ps.save("getting_started_solution.psz", solver)
loaded = ps.load("getting_started_solution.psz")
print("Lift coefficient after reload:", loaded.airplanes[0].liftCoefficient_W)

INFO    |_serialization                              |Saving SteadyHorseshoeVortexLatticeMethodSolver to getting_started_solution.psz
INFO    |_serialization                              |Saved SteadyHorseshoeVortexLatticeMethodSolver to getting_started_solution.psz (126240 bytes)
INFO    |_serialization                              |Loading from getting_started_solution.psz
INFO    |_serialization                              |Loaded SteadyHorseshoeVortexLatticeMethodSolver from getting_started_solution.psz


Lift coefficient after reload: 0.5548759749558987


## Summary

You have now run your first Ptera Software simulation. The object model is
similar across the solvers, so the same pattern applies to steady ring
VLM, unsteady ring VLM, aeroelastic, and free flight simulations:

```text
Airfoil -> WingCrossSection -> Wing -> Airplane
OperatingPoint -> Problem -> Solver -> run()
```

To go further, look at the example scripts in `examples/` and the API
reference on the documentation site.
